# Data Acquisition II — Web Scraping

## Learning goals

By the end of this lesson, you should be able to:

- explain when web scraping may be appropriate;
- distinguish between HTML, CSS, and JavaScript at a basic level;
- inspect the HTML structure of a webpage;
- identify tags, attributes, classes, and nested elements;
- download static HTML with `requests`;
- parse HTML with `BeautifulSoup`;
- extract text and HTML attributes;
- iterate over repeated page components;
- follow links from a listing page to a detail page;
- handle simple pagination;
- organize scraped data in a Pandas DataFrame;
- recognize situations where `requests + BeautifulSoup` is not enough;
- apply basic responsible-scraping checks before collecting data.

The central question is:

> **We need external data, but there is no suitable API. Is the information available on a webpage we can responsibly scrape?**

## 1. From APIs to Web Scraping

In the previous lesson, we used an API to obtain structured external data.

But sometimes there is no suitable API.

```text
We need external data
        │
        ▼
Is it already available
as a downloadable dataset?
     │             │
    YES            NO
     │             │
     ▼             ▼
Download it    Is there a suitable API?
                  │            │
                 YES           NO
                  │             │
                  ▼             ▼
                 API      Is the information
                          available on a webpage?
                                │
                               YES
                                │
                                ▼
                          Web Scraping
```

Web scraping means:

1. requesting a webpage;
2. reading its HTML;
3. locating the elements that contain the information we need;
4. extracting that information into a structured form.

## 2. Before scraping: should we scrape this website?

Just because information is visible in a browser does **not** automatically mean we should scrape it.

Before starting:

1. **Check whether an API or downloadable dataset already exists.**
2. **Read the site's Terms of Service when relevant.**
3. **Check `robots.txt`.**
4. **Avoid collecting private or sensitive information.**
5. **Do not send requests unnecessarily fast.**
6. **Do not attempt to bypass access controls, CAPTCHAs, logins, or technical restrictions.**
7. **Expect websites to change.**

A typical `robots.txt` URL is:

```text
https://www.example.com/robots.txt
```

`robots.txt` provides instructions for automated crawlers, but it is not a substitute for checking the site's terms or applicable rules.

## 3. Our practice website

For this lesson we will use:

https://quotes.toscrape.com/

**Quotes to Scrape** is part of a web-scraping sandbox created specifically for scraping practice.

We will use its normal static endpoint, which includes:

- repeated quote components;
- author links;
- tags;
- pagination.

This is a better teaching target than an arbitrary commercial website because its structure exists specifically for scraping practice.

## 4. How does a webpage work?

Three technologies are especially important:

### HTML — structure and content

HTML defines elements such as:

```html
<h1>Heading</h1>
<p>A paragraph</p>
<a href="/about">About</a>
```

### CSS — presentation

CSS controls how HTML elements look.

Classes are also often useful when identifying elements for scraping.

### JavaScript — behavior and dynamic content

JavaScript can modify a webpage after the initial HTML has loaded.

For this lesson we will scrape **static HTML**.

```text
requests
   ↓
HTML returned by server
   ↓
BeautifulSoup
   ↓
extract information
```

If the information is created only later by JavaScript, `requests` may not see it.

## 5. The minimum HTML you need

You do **not** need to learn web development to scrape a simple page.

You mainly need three ideas.

### 1. Tags

HTML content is placed inside tags:

```html
<p>Hello</p>
```

### 2. Attributes

Tags can contain attributes:

```html
<a class="author-link" href="/author/Ada-Lovelace">
    About
</a>
```

Useful attributes include:

- `class`
- `id`
- `href`
- `src`
- `title`

### 3. Nesting

HTML is hierarchical:

```html
<div class="quote">
    <span class="text">A quote</span>
    <small class="author">An author</small>
</div>
```

The `<span>` and `<small>` elements are inside the `<div>`.

### A useful mental model

HTML behaves like a tree:

```text
html
└── body
    ├── div.quote
    │   ├── span.text
    │   └── small.author
    └── div.quote
        ├── span.text
        └── small.author
```

Our task is to identify the branch containing the information we need.

### 💡 Quick Check 1 — Complete the code

Given this HTML:

```html
<div class="product">
    <h2>Wireless Mouse</h2>
    <a href="/products/mouse">Details</a>
</div>
```

Complete the comments.

In [ ]:
# "div", "h2" and "a" are HTML ________.
# "class" and "href" are HTML ________.
# The h2 and a elements are ________ inside the div element.

## 6. Inspect before writing scraping code

Before using Python, inspect the webpage in your browser.

A practical workflow is:

```text
1. Open the webpage
        ↓
2. Find one example of the information
        ↓
3. Right-click → Inspect
        ↓
4. Identify its HTML tag
        ↓
5. Identify a useful class / attribute
        ↓
6. Look for the repeated parent component
```

For example, on Quotes to Scrape, inspect:

- one quote;
- its author;
- its tags;
- the **about** link.

The browser's developer tools are often the most important debugging tool in web scraping.

## 7. Required libraries

We will use:

- `requests` → download the webpage;
- `beautifulsoup4` → parse and search the HTML;
- `pandas` → structure the extracted data.

First try:

```python
import requests
from bs4 import BeautifulSoup
import pandas as pd
```

If `BeautifulSoup` is missing, install it with Conda.

### macOS / Linux

Open a new **Terminal** and run:

```bash
conda install -y anaconda::beautifulsoup4
```

### Windows

Open a new **Anaconda PowerShell Prompt** and run:

```powershell
conda install -y anaconda::beautifulsoup4
```

The `-y` option automatically confirms the installation.

## 8. Step 1 — Request the webpage

This should feel familiar from the API lesson.

We still use:

```python
requests.get(...)
```

But the response is different.

```text
API request                 Webpage request
     │                            │
     ▼                            ▼
   JSON                          HTML
     │                            │
     ▼                            ▼
response.json()            BeautifulSoup(...)
```

In [ ]:
import requests

url = "https://quotes.toscrape.com/"

response = requests.get(url)

print(response.status_code)
print(response.headers.get("Content-Type"))

Always check that the request succeeded before parsing the response.

In [ ]:
response.raise_for_status()

The HTML returned by the server is available through `response.text`.

In [ ]:
print(response.text[:1000])

### 💡 Quick Check 2 — Complete the code

Request the page and stop if the HTTP response indicates an error.

In [ ]:
url = "https://quotes.toscrape.com/"

response = requests._____(url)

response.________________()

## 9. Step 2 — Parse HTML with BeautifulSoup

Raw HTML is just text.

BeautifulSoup parses that text into a tree that we can search.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print(type(soup))

We can make the HTML easier to inspect with `prettify()`.

In [ ]:
print(soup.prettify()[:1500])

## 10. Finding HTML elements

BeautifulSoup provides several ways to locate elements.

For beginners, two methods are especially useful:

### `find()`

Returns the **first** matching element.

```python
soup.find("tag")
```

### `find_all()`

Returns **all** matching elements.

```python
soup.find_all("tag")
```

In [ ]:
first_quote = soup.find("div", class_="quote")

display(first_quote)

Why `class_` instead of `class`?

Because `class` is a reserved keyword in Python, BeautifulSoup uses:

```python
class_="..."
```

### 💡 Quick Check 3 — Complete the code

Find all `<div>` elements whose class is `quote`.

In [ ]:
quote_blocks = soup.________(
    "div",
    class_=________
)

print(len(quote_blocks))

## 11. Extracting text from an element

Once we have found an element, `.get_text()` extracts its text.

`strip=True` removes unnecessary whitespace.

In [ ]:
quote_text = first_quote.find(
    "span",
    class_="text"
).get_text(strip=True)

author = first_quote.find(
    "small",
    class_="author"
).get_text(strip=True)

print(quote_text)
print(author)

A useful habit is:

> **First isolate one complete repeated component. Then extract its fields from inside that component.**

This is safer than extracting all authors and all quotes independently and assuming the lists always stay aligned.

### 💡 Quick Check 4 — Complete the code

Extract the author from `first_quote`.

In [ ]:
author = (
    first_quote
    .find("small", class_=________)
    .________(strip=True)
)

print(author)

## 12. Extracting attributes

Sometimes the information we need is **inside an attribute**, not between the tags.

For example:

```html
<a href="/author/Albert-Einstein">about</a>
```

The visible text is:

```text
about
```

but the useful information is the value of `href`.

Use `.get()`:

In [ ]:
about_link = first_quote.find("a")

relative_url = about_link.get("href")

print(relative_url)

The website returns a **relative URL**:

```text
/author/Albert-Einstein
```

We can convert it to an absolute URL safely with `urljoin`.

In [ ]:
from urllib.parse import urljoin

author_url = urljoin(
    url,
    relative_url
)

print(author_url)

### 💡 Quick Check 5 — Complete the code

Extract the `href` attribute and create the complete URL.

In [ ]:
link = first_quote.find("a")

relative_url = link._____("href")

author_url = urljoin(
    url,
    ________
)

print(author_url)

## 13. Repeated elements → structured records

A webpage often contains repeated components.

Quotes to Scrape has several:

```html
<div class="quote"> ... </div>
<div class="quote"> ... </div>
<div class="quote"> ... </div>
```

We can:

1. find every quote component;
2. iterate through them;
3. extract the same fields from each component;
4. store each result as a dictionary.

In [ ]:
quote_blocks = soup.find_all(
    "div",
    class_="quote"
)

records = []

for block in quote_blocks:
    quote = block.find(
        "span",
        class_="text"
    ).get_text(strip=True)

    author = block.find(
        "small",
        class_="author"
    ).get_text(strip=True)

    tags = [
        tag.get_text(strip=True)
        for tag in block.find_all("a", class_="tag")
    ]

    about_href = block.find("a").get("href")

    records.append({
        "quote": quote,
        "author": author,
        "tags": tags,
        "author_url": urljoin(url, about_href)
    })

records[:2]

The list of dictionaries can now become a DataFrame.

In [ ]:
quotes_df = pd.DataFrame(records)

display(quotes_df.head())

This is the complete transformation:

```text
Webpage
  ↓
HTML
  ↓
BeautifulSoup tree
  ↓
Repeated components
  ↓
Python dictionaries
  ↓
Pandas DataFrame
```

### 💡 Quick Check 6 — Complete the code

Create a DataFrame from the `records` list.

In [ ]:
quotes_df = pd.________(________)

display(quotes_df.head())

## 14. Listing page → detail page

Sometimes the listing page does not contain all the information we need.

A common website structure is:

```text
LISTING PAGE
product / quote / article cards
        │
        ├── detail link
        ├── detail link
        └── detail link
              ↓
        DETAIL PAGE
        additional fields
```

In Quotes to Scrape, the quote listing contains a link to an author's detail page.

Let's request one of those pages.

In [ ]:
author_response = requests.get(
    records[0]["author_url"]
)

author_response.raise_for_status()

author_soup = BeautifulSoup(
    author_response.text,
    "html.parser"
)

print(author_response.url)

Inspect the author page and extract additional fields.

In [ ]:
birth_date = author_soup.find(
    "span",
    class_="author-born-date"
).get_text(strip=True)

birth_location = author_soup.find(
    "span",
    class_="author-born-location"
).get_text(strip=True)

print(birth_date)
print(birth_location)

This pattern is important:

```text
request listing page
        ↓
extract detail-page URL
        ↓
request detail page
        ↓
extract additional information
```

Be careful: following every detail link means making additional HTTP requests.

### 💡 Quick Check 7 — Complete the code

Request the first author's detail page and parse it.

In [ ]:
detail_url = records[0]["author_url"]

detail_response = requests._____(detail_url)

detail_response.________________()

detail_soup = BeautifulSoup(
    detail_response._____,
    "html.parser"
)

## 15. Pagination

The first page does not contain every quote.

At the bottom of the page there is a **Next** link.

Instead of guessing the URL pattern, we can extract that link from the HTML.

In [ ]:
next_link = soup.find(
    "li",
    class_="next"
)

display(next_link)

The `<li>` contains an `<a>` with the URL of the next page.

In [ ]:
next_href = next_link.find("a").get("href")

next_url = urljoin(
    url,
    next_href
)

print(next_url)

A simple pagination loop can continue until there is no Next link.

For teaching purposes, we will limit the number of pages so that we do not make unnecessary requests.

In [ ]:
page_url = "https://quotes.toscrape.com/"
all_records = []

max_pages = 3

for page_number in range(max_pages):
    response = requests.get(page_url)
    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    blocks = soup.find_all(
        "div",
        class_="quote"
    )

    for block in blocks:
        all_records.append({
            "quote": block.find(
                "span",
                class_="text"
            ).get_text(strip=True),

            "author": block.find(
                "small",
                class_="author"
            ).get_text(strip=True)
        })

    next_element = soup.find(
        "li",
        class_="next"
    )

    if next_element is None:
        break

    page_url = urljoin(
        page_url,
        next_element.find("a").get("href")
    )

quotes_pages_df = pd.DataFrame(all_records)

display(quotes_pages_df.head())
print("Rows:", len(quotes_pages_df))

Notice the logic:

```text
request page
    ↓
extract records
    ↓
is there a Next link?
   /            \
 YES             NO
  ↓               ↓
build URL         stop
  ↓
request next page
```

This approach is often more robust than manually generating page numbers.

### 💡 Quick Check 8 — Complete the code

Complete the stopping condition.

In [ ]:
next_element = soup.find(
    "li",
    class_="next"
)

if next_element is ________:
    ________

## 16. Static vs dynamic content

`requests` downloads the HTML returned directly by the server.

That works well for **static pages**.

But modern pages may load information later using JavaScript.

```text
Server returns HTML
        ↓
Browser loads page
        ↓
JavaScript runs
        ↓
additional content appears
```

If the data only appears after JavaScript runs:

```python
requests.get(...)
```

may not contain the information you can see in the browser.

Possible browser-automation tools for dynamic websites include:

- [Selenium — Python documentation](https://www.selenium.dev/selenium/docs/api/py/)
- [Playwright — Python documentation](https://playwright.dev/python/docs/intro)

These tools launch or control a real browser, so JavaScript can execute before the data is extracted.

They are **outside the scope of this introductory lesson**.

For now, the important diagnostic question is:

> **Is the information I need present in the HTML returned by `requests`?**

## 17. A practical scraping workflow

Before writing a large scraping script, work incrementally:

```text
1. Define the data you need
        ↓
2. Check for API / downloadable data
        ↓
3. Check whether scraping is appropriate
        ↓
4. Inspect one page in the browser
        ↓
5. Request the page
        ↓
6. Check HTTP status
        ↓
7. Parse HTML
        ↓
8. Find ONE repeated component
        ↓
9. Extract ONE record correctly
        ↓
10. Generalize with a loop
        ↓
11. Follow detail links if necessary
        ↓
12. Add pagination if necessary
        ↓
13. Build a DataFrame
        ↓
14. Verify the result
```

Do not start by trying to scrape the entire website.

> **Get one record right first. Then generalize.**

## 18. Final Challenge — Complete the scraper

Use the first page of Quotes to Scrape.

Create a DataFrame containing:

- `quote`
- `author`
- `tags`
- `author_url`

Complete the missing code.

In [ ]:
url = "https://quotes.toscrape.com/"

# 1. Request
response = requests._____(url)
response.________________()

# 2. Parse
soup = BeautifulSoup(
    response._____,
    "html.parser"
)

# 3. Find repeated components
quote_blocks = soup.________(
    "div",
    class_=________
)

records = []

for block in quote_blocks:

    quote = block.find(
        "span",
        class_="text"
    ).________(strip=True)

    author = block.find(
        "small",
        class_="author"
    ).________(strip=True)

    tags = [
        tag.get_text(strip=True)
        for tag in block.________(
            "a",
            class_="tag"
        )
    ]

    relative_url = block.find("a").________("href")

    records.append({
        "quote": quote,
        "author": author,
        "tags": tags,
        "author_url": urljoin(url, relative_url)
    })

# 4. Structure data
quotes = pd.________(records)

display(quotes.head())

## Summary

Today you learned a basic static web-scraping pipeline:

```text
Inspect
   ↓
Request
   ↓
Parse
   ↓
Find
   ↓
Extract
   ↓
Follow links / paginate when needed
   ↓
Structure
   ↓
Verify
```

You practiced:

- HTML tags, attributes, classes, and nesting;
- browser inspection;
- `requests.get()`;
- HTTP status checking;
- `BeautifulSoup`;
- `find()` and `find_all()`;
- `.get_text()`;
- extracting attributes with `.get()`;
- repeated components;
- detail-page links;
- simple pagination;
- creating Pandas DataFrames.

### Further exploration

More advanced scraping may involve:

- CSS selectors with `select()` / `select_one()`;
- sessions and cookies;
- forms and authentication;
- dynamic JavaScript content;
- [Selenium — Python documentation](https://www.selenium.dev/selenium/docs/api/py/);
- [Playwright — Python documentation](https://playwright.dev/python/docs/intro);
- [Scrapy — documentation](https://docs.scrapy.org/en/latest/), a complete framework for larger crawling and scraping projects;
- rate limiting and retries;
- more sophisticated crawling strategies.

> **Scrapy is not a browser automation tool by itself.**  
> For JavaScript-heavy pages, first try to identify the underlying data request. If browser rendering is genuinely required, Scrapy can be combined with browser automation tools such as Playwright.

These tools are useful, but the fundamental process remains:

> **Inspect the structure → retrieve the content → extract only what you need.**